# Training Tricks That Matter

**Time: ~35 minutes**

Getting a PINN to converge isn't always easy. This notebook covers the practical techniques that make the difference between a PINN that works and one that doesn't:

1. **Loss weighting** — balancing competing objectives
2. **Spectral bias** — why PINNs struggle with high frequencies
3. **The Ansatz trick** — embedding known structure
4. **Learning rate scheduling** — Adam → L-BFGS or decay
5. **Input normalization** — keeping inputs in a good range
6. **Best-model saving** — don't lose your best result

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

## 1. Loss Weighting

A PINN has multiple loss terms (physics, IC, BC). If one dominates, the others are ignored.

**Problem setup:** `u' = -u`, `u(0) = 1`, but with the IC severely underweighted.

In [ ]:
def make_model():
    return nn.Sequential(
        nn.Linear(1, 32), nn.Tanh(),
        nn.Linear(32, 32), nn.Tanh(),
        nn.Linear(32, 1),
    )

t_phys = torch.linspace(0, 2, 100).unsqueeze(1).requires_grad_(True)
t_ic = torch.zeros(1, 1)

def train_with_weights(w_ic, n_epochs=5000):
    model = make_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(n_epochs):
        optimizer.zero_grad()
        u = model(t_phys)
        du_dt = torch.autograd.grad(u, t_phys, torch.ones_like(u), create_graph=True)[0]
        loss_phys = torch.mean((du_dt + u)**2)
        loss_ic = (model(t_ic) - 1.0)**2
        (loss_phys + w_ic * loss_ic).backward()
        optimizer.step()
    return model

# Train with different IC weights
models = {}
for w in [0.1, 1.0, 10.0, 100.0]:
    models[w] = train_with_weights(w)
    print(f"w_ic = {w:5.1f} | u(0) = {models[w](t_ic).item():.4f} (should be 1.0)")

In [ ]:
t_test = torch.linspace(0, 2, 200).unsqueeze(1)
u_exact = np.exp(-t_test.numpy())

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(t_test.numpy(), u_exact, 'k-', linewidth=2, label='Exact')
colors = ['red', 'orange', 'green', 'blue']
for (w, m), c in zip(models.items(), colors):
    with torch.no_grad():
        ax.plot(t_test.numpy(), m(t_test).numpy(), '--', color=c, label=f'w_ic = {w}')
ax.set_xlabel('t'); ax.set_ylabel('u(t)')
ax.set_title('Effect of IC Weight on Solution')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("Too low: the IC isn't satisfied. Too high: optimization focuses only on t=0.")
print("Rule of thumb: IC/BC weights of 10-100x work well as a starting point.")

## 2. Spectral Bias

Neural networks learn **low-frequency** components first — this is called spectral bias. For solutions with high-frequency oscillations (like a damped oscillator), plain PINNs converge painfully slowly.

Let's see this in action with a fast-oscillating solution.

In [ ]:
# Target: u(t) = exp(-t) * cos(20*t)  — damped high-frequency oscillation
omega = 20.0

t_plot = np.linspace(0, 2, 500)
u_target = np.exp(-t_plot) * np.cos(omega * t_plot)

plt.figure(figsize=(10, 4))
plt.plot(t_plot, u_target, 'b-')
plt.xlabel('t'); plt.ylabel('u(t)')
plt.title(f'Target: exp(-t) * cos({omega}t) — try learning this with a plain network!')
plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Try to fit this with a plain network (supervised, not PINN — to isolate spectral bias)
t_train = torch.linspace(0, 2, 500).unsqueeze(1)
u_train = torch.exp(-t_train) * torch.cos(omega * t_train)

model_plain = nn.Sequential(
    nn.Linear(1, 64), nn.Tanh(),
    nn.Linear(64, 64), nn.Tanh(),
    nn.Linear(64, 64), nn.Tanh(),
    nn.Linear(64, 1),
)

optimizer = torch.optim.Adam(model_plain.parameters(), lr=1e-3)
for epoch in range(10000):
    optimizer.zero_grad()
    loss = torch.mean((model_plain(t_train) - u_train)**2)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    pred_plain = model_plain(t_train).numpy()

plt.figure(figsize=(10, 4))
plt.plot(t_train.numpy(), u_train.numpy(), 'b-', label='Target', alpha=0.7)
plt.plot(t_train.numpy(), pred_plain, 'r--', label='Plain network (10k epochs)', alpha=0.7)
plt.xlabel('t'); plt.ylabel('u(t)')
plt.title('Spectral Bias: plain network struggles with high-frequency content')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. The Ansatz Trick

If you **know the solution oscillates** at frequency `omega`, embed that structure analytically:

```
u(t) = A(t) * cos(omega * t) + B(t) * sin(omega * t)
```

where `A(t)` and `B(t)` are the **slowly varying envelopes** learned by the network. The network no longer needs to learn the oscillation — just the envelope.

In [ ]:
class AnsatzModel(nn.Module):
    def __init__(self, omega):
        super().__init__()
        self.omega = omega
        self.backbone = nn.Sequential(
            nn.Linear(1, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 2),  # outputs [A(t), B(t)]
        )

    def forward(self, t):
        envelopes = self.backbone(t)
        A, B = envelopes[:, 0:1], envelopes[:, 1:2]
        return A * torch.cos(self.omega * t) + B * torch.sin(self.omega * t)

model_ansatz = AnsatzModel(omega)
optimizer = torch.optim.Adam(model_ansatz.parameters(), lr=1e-3)

for epoch in range(5000):  # Half the epochs!
    optimizer.zero_grad()
    loss = torch.mean((model_ansatz(t_train) - u_train)**2)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    pred_ansatz = model_ansatz(t_train).numpy()

plt.figure(figsize=(10, 4))
plt.plot(t_train.numpy(), u_train.numpy(), 'b-', label='Target', alpha=0.7)
plt.plot(t_train.numpy(), pred_plain, 'r--', label='Plain (10k epochs)', alpha=0.5)
plt.plot(t_train.numpy(), pred_ansatz, 'g-', label='Ansatz (5k epochs)', linewidth=2)
plt.xlabel('t'); plt.ylabel('u(t)')
plt.title('Ansatz defeats spectral bias — better fit in half the epochs')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Input Normalization

Neural networks work best when inputs are in `[-1, 1]`. If your domain is `x in [0, 100]` and `t in [0, 0.001]`, normalize them:

```python
x_norm = 2 * (x - x_min) / (x_max - x_min) - 1  # maps to [-1, 1]
```

For parametric PINNs, this is **critical** — parameters spanning different scales (e.g., frequency in [20, 100] and damping in [0.1, 4]) must be normalized independently.

## 5. Learning Rate and Optimization

The typical strategy:

1. **Start with Adam** (lr=1e-3) — robust, handles noisy gradients well
2. **Reduce lr** when the loss plateaus (ReduceLROnPlateau or cosine annealing)
3. Optionally **switch to L-BFGS** for final refinement — quasi-Newton method, better for smooth landscapes

Let's compare a fixed lr vs a decaying schedule.

In [ ]:
# Training u' = -u with fixed lr vs cosine decay
def train_exponential_decay(use_scheduler=False, n_epochs=5000):
    model = make_model()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=n_epochs
    ) if use_scheduler else None

    history = []
    for _ in range(n_epochs):
        optimizer.zero_grad()
        u = model(t_phys)
        du = torch.autograd.grad(u, t_phys, torch.ones_like(u), create_graph=True)[0]
        loss = torch.mean((du + u)**2) + 10 * (model(t_ic) - 1)**2
        loss.backward()
        optimizer.step()
        if scheduler: scheduler.step()
        history.append(loss.item())
    return history

hist_fixed = train_exponential_decay(use_scheduler=False)
hist_cosine = train_exponential_decay(use_scheduler=True)

plt.figure(figsize=(8, 5))
plt.semilogy(hist_fixed, label='Fixed lr=1e-3', alpha=0.8)
plt.semilogy(hist_cosine, label='Cosine annealing', alpha=0.8)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Learning Rate Schedule Comparison')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Best-Model Saving

Training loss is noisy. The model at epoch 8000 might be better than at epoch 10000. **Always track the best model:**

```python
# The PINNTrainer in this repo does this automatically:
trainer.train(
    ...,
    save_best=run_dir / "best_model.pt",  # saves best weights
    restore_best=True,                      # restores them at end
)
```

This ensures the final checkpoint contains the best model found during training, not the last.

## Summary: The Tricks That Matter

| Trick | When to use | Impact |
|-------|------------|--------|
| **IC/BC weighting** | Always | High — wrong weights = wrong solution |
| **Ansatz** | Known frequency/structure | Very high — can be 100x faster |
| **Input normalization** | Multi-scale inputs | High — prevents gradient imbalance |
| **LR scheduling** | Long training runs | Medium — helps final convergence |
| **Best-model saving** | Always | Medium — prevents losing the best result |
| **Gradient clipping** | Unstable training | Medium — prevents exploding gradients |
| **More collocation points** | Undersampled regions | Medium — diminishing returns |

The first three are the most impactful. If your PINN isn't converging, check these first.

## What's Next

**Notebook 07** covers parametric PINNs (one model for a whole parameter family) and inverse problems (inferring unknown parameters from data) — two of the most powerful PINN applications.